# SemanticRAG vs. VKG — Query-Agent Evaluation Comparison

Pure comparator: loads the k=3 mean results from **notebook 2** (SemanticRAG /
metadata query agent) and **notebook 6** (VKG / ontology query agent) and places
them side-by-side. No agents are invoked here — run notebooks 2 and 6 first.

## 1. Setup

In [1]:
import glob
import json
import os
import re

import boto3
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv(dotenv_path='.env', override=False)
os.environ.setdefault('AWS_PROFILE', 'huthmac+demo')
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')

session = boto3.Session(profile_name=os.environ['AWS_PROFILE'])
region = session.region_name or 'us-east-1'
account_id = session.client('sts').get_caller_identity()['Account']
print(f'✓ AWS ...{account_id[-4:]}, region {region}')

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)


def _redact_account_ids(obj):
    """Recursively mask AWS account IDs in ARN strings to their last 4 digits (********NNNN)."""
    if isinstance(obj, dict):
        return {k: _redact_account_ids(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_redact_account_ids(v) for v in obj]
    if isinstance(obj, str):
        return re.sub(r'(\d{8})(\d{4})', r'********\2', obj)
    return obj

✓ AWS ...4087, region us-east-1


## 2. Load k-run means from notebooks 2 and 6

In [2]:
def load_latest_kmean(prefix: str, *, label: str) -> dict:
    """Load the most recent results/{prefix}_*.json produced by a k-run eval notebook.

    Raises FileNotFoundError if no file exists, so a missing upstream run is
    obvious rather than silently producing an incomplete comparison table.
    """
    matches = sorted(glob.glob(f'../data/eval/results/{prefix}_*.json'))
    if not matches:
        raise FileNotFoundError(
            f'No {prefix}_*.json in ../data/eval/results/. '
            f'Run notebook 2 (SemanticRAG) or notebook 6 (VKG) first.'
        )
    path = matches[-1]
    payload = json.load(open(path))
    print(f'  ♻ [{label}] {os.path.basename(path)} '
          f'(eval_k={payload.get("eval_k")}, scores={payload.get("mean_scores")})')
    return {
        'label': label,
        'mean_scores': payload.get('mean_scores', {}) or {},
        'std_scores': payload.get('std_scores', {}) or {},
        'agent_perf_mean': dict(payload.get('agent_perf_mean', {}) or {}),
        # per_scenario_agent carries per-query token/latency + whether each query
        # actually emitted SQL — the basis for a per-query (not whole-batch) token
        # comparison that stays fair when one arm degrades on half the questions.
        'per_scenario_agent': payload.get('per_scenario_agent', {}) or {},
        'per_scenario_goal_mean': payload.get('per_scenario_goal_mean', {}) or {},
        'eval_id': payload.get('eval_id'),
        'eval_k': payload.get('eval_k'),
        'source_file': os.path.basename(path),
    }


def avg_tokens_per_query(kmean: dict) -> dict:
    """Mean per-query agent token cost, restricted to queries that actually executed SQL.

    Whole-batch `total_tokens` is misleading across arms: an arm that degrades on
    half the questions (no SQL emitted) burns fewer tokens for a worse outcome, so a
    lower total looks like a win. Averaging `per_scenario_agent[*].avg_total_tokens`
    over only the scenarios that emitted SQL gives an apples-to-apples cost-per-answered
    query. Returns the mean plus the executed/total scenario counts (so the denominator
    is never hidden).

    Parameters:
        kmean: a loaded k-run mean payload (must carry ``per_scenario_agent``).
    Returns:
        {'avg_token_per_query': float|None, 'executed': int, 'total': int}.
    """
    psa = kmean.get('per_scenario_agent', {}) or {}
    executed_tokens = [s.get('avg_total_tokens') for s in psa.values()
                       if s.get('emitted_sql') and isinstance(s.get('avg_total_tokens'), (int, float))]
    return {
        'avg_token_per_query': round(sum(executed_tokens) / len(executed_tokens))
        if executed_tokens else None,
        'executed': len(executed_tokens),
        'total': len(psa),
    }


kmean_a = load_latest_kmean('metadata_query_kmean_eval', label='semantic-rag')
kmean_b = load_latest_kmean('ontology_query_kmean_eval', label='vkg')

  ♻ [semantic-rag] metadata_query_kmean_eval_20260701_105036.json (eval_k=3, scores={'GoalSuccess': 0.75, 'FinalAnswerFaithfulness': 0.75, 'SqlGrounded': 1.0})
  ♻ [vkg] ontology_query_kmean_eval_20260701_114640.json (eval_k=3, scores={'GoalSuccess': 0.71, 'FinalAnswerFaithfulness': 0.71, 'SqlGrounded': 1.0})


## 3. Compare

In [ ]:
_EVALS = [
    'GoalSuccess',
    'FinalAnswerFaithfulness',
    'SqlGrounded',
]


def summarize_kmean(kmean: dict) -> dict:
    scores = kmean.get('mean_scores', {}) or {}
    std = kmean.get('std_scores', {}) or {}
    perf = kmean.get('agent_perf_mean', {}) or {}
    row = {'path': kmean.get('label'), 'eval_k': kmean.get('eval_k')}
    for ev in _EVALS:
        mean_val = scores.get(ev)
        std_val = std.get(ev)
        if mean_val is not None and std_val is not None:
            row[ev] = f'{mean_val:.2f} ± {std_val:.2f}'
        else:
            row[ev] = mean_val
    row['avg_latency_s'] = perf.get('avg_wall_clock_s')
    # Per-query token cost over the queries that actually executed SQL — NOT the
    # whole-batch total. A batch total flatters an arm that degrades on half the
    # questions (it emits no SQL on those, so its total drops for a worse outcome);
    # the per-query mean over executed queries is the fair cross-arm cost metric.
    tpq = avg_tokens_per_query(kmean)
    row['avg_token_per_query'] = tpq['avg_token_per_query']
    row['queries_with_sql'] = f"{tpq['executed']}/{tpq['total']}"
    return row


comparison = pd.DataFrame([summarize_kmean(kmean_a), summarize_kmean(kmean_b)])
print('=== SemanticRAG (nb2) vs VKG (nb6) — k-run mean ± std ===')
print('(avg_token_per_query = mean agent tokens over the queries that executed SQL; '
      'queries_with_sql shows that denominator)')
display(comparison)

# Save result
os.makedirs('../data/eval/results', exist_ok=True)
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
out = f'../data/eval/results/semantic_rag_vs_vkg_kmean_{ts}.json'
with open(out, 'w') as f:
    json.dump(
        _redact_account_ids({
            'eval_k': {'semantic-rag': kmean_a.get('eval_k'), 'vkg': kmean_b.get('eval_k')},
            'sources': {'semantic-rag': kmean_a.get('source_file'), 'vkg': kmean_b.get('source_file')},
            'eval_ids': {'semantic-rag': kmean_a.get('eval_id'), 'vkg': kmean_b.get('eval_id')},
            'kmean_a': kmean_a,
            'kmean_b': kmean_b,
            'comparison': comparison.to_dict(orient='records'),
        }),
        f, indent=2, default=str,
    )
print(f'✓ Wrote {out}')

# Headline
a_s = kmean_a.get('mean_scores', {}) or {}
b_s = kmean_b.get('mean_scores', {}) or {}
a_gsr = a_s.get('GoalSuccess')
b_gsr = b_s.get('GoalSuccess')
if a_gsr is not None and b_gsr is not None:
    winner = 'vkg' if b_gsr > a_gsr else ('semantic-rag' if a_gsr > b_gsr else 'tie')
    a_p = kmean_a.get('agent_perf_mean', {})
    b_p = kmean_b.get('agent_perf_mean', {})
    a_tpq = avg_tokens_per_query(kmean_a)
    b_tpq = avg_tokens_per_query(kmean_b)
    display(Markdown(
        '### Headline (k-run mean)\n'
        f'- **Runs:** semantic-rag k={kmean_a.get("eval_k")} (nb2) · vkg k={kmean_b.get("eval_k")} (nb6)\n'
        f'- **GoalSuccessRate:** semantic-rag {a_gsr:.1%} vs vkg {b_gsr:.1%} → **{winner}**\n'
        f'- **FinalAnswerFaithfulness:** {a_s.get("FinalAnswerFaithfulness")} vs {b_s.get("FinalAnswerFaithfulness")}\n'
        f'- **SqlGrounded:** {a_s.get("SqlGrounded")} vs {b_s.get("SqlGrounded")}\n'
        f'- **Avg latency:** {a_p.get("avg_wall_clock_s")}s vs {b_p.get("avg_wall_clock_s")}s\n'
        f'- **Avg tokens / query** (over queries that executed SQL): '
        f'{a_tpq["avg_token_per_query"]} ({a_tpq["executed"]}/{a_tpq["total"]} queries) vs '
        f'{b_tpq["avg_token_per_query"]} ({b_tpq["executed"]}/{b_tpq["total"]} queries)\n'
        '\n_Per-query tokens (not the batch total) is the fair cost metric: an arm that '
        'degrades on half the questions emits no SQL there, so its batch total understates '
        'its real cost-per-answered-query. VKG SqlGrounded is also affected by span-harvest '
        'noise; judge VKG primarily by GoalSuccessRate._'
    ))
else:
    print('⚠ One or both arms missing scores — run notebooks 2 and 6 first.')

=== SemanticRAG (nb2) vs VKG (nb6) — k-run mean ± std ===
(avg_token_per_query = mean agent tokens over the queries that executed SQL; queries_with_sql shows that denominator)


,path,eval_k,GoalSuccess,FinalAnswerFaithfulness,SqlGrounded,avg_latency_s,avg_token_per_query,queries_with_sql
0,semantic-rag,3,0.75 ± 0.00,0.75 ± 0.00,1.00 ± 0.00,26.2933,99749,9/16
1,vkg,3,0.71 ± 0.03,0.71 ± 0.03,1.00 ± 0.00,40.9433,112580,11/16


✓ Wrote ../data/eval/results/semantic_rag_vs_vkg_kmean_20260701_120646.json


### Headline (k-run mean)
- **Runs:** semantic-rag k=3 (nb2) · vkg k=3 (nb6)
- **GoalSuccessRate:** semantic-rag 75.0% vs vkg 71.0% → **semantic-rag**
- **FinalAnswerFaithfulness:** 0.75 vs 0.71
- **SqlGrounded:** 1.0 vs 1.0
- **Avg latency:** 26.2933s vs 40.9433s
- **Avg tokens / query** (over queries that executed SQL): 99749 (9/16 queries) vs 112580 (11/16 queries)

_Per-query tokens (not the batch total) is the fair cost metric: an arm that degrades on half the questions emits no SQL there, so its batch total understates its real cost-per-answered-query. VKG SqlGrounded is also affected by span-harvest noise; judge VKG primarily by GoalSuccessRate._

: 